# llama.cpp on Kaggle (GPU)
1. Enable **GPU** accelerator (T4) + **Internet** in notebook settings.
2. Replace `YOUR_GITHUB_URL` and `MODEL_FILE` below, then run cells top to bottom.
3. Sessions are ephemeral — re-run this notebook each time. Keep 1–2 models per session (disk is ~20–30GB).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv
git clone YOUR_GITHUB_URL llama-setup
cd llama-setup
chmod +x setup.sh start-server.sh
./setup.sh

Download **one** model into `models/` (HuggingFace GGUF repo). Example pattern — set `REPO_ID` and the wanted filename:

In [ ]:
REPO_ID = "unsloth/Qwen3-30B-A3B-GGUF"  # <-- change me
FILE = "Qwen3-30B-A3B-Q4_K_M.gguf"       # <-- change me

from huggingface_hub import snapshot_download
snapshot_download(repo_id=REPO_ID, local_dir="llama-setup/models",
                  allow_patterns=[FILE])
print("saved:", FILE)

Swap in the T4-tuned config, add a matching line for the new model if needed (`filename|ctx|ngl|kv`), then launch in the background and wait for health:

In [ ]:
%%bash
cd llama-setup
cp config/models.conf.kaggle-t4 config/models.conf
MODEL_FILE="Qwen3-30B-A3B-Q4_K_M.gguf"  # <-- must match FILE above
nohup ./start-server.sh "$MODEL_FILE" 18123 127.0.0.1 > logs/kaggle-server.log 2>&1 &
for i in $(seq 1 30); do curl -sf http://127.0.0.1:18123/health && break; sleep 10; done

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:18123/v1", api_key="sk-llama")
r = client.chat.completions.create(
    model="Local Model",
    messages=[{"role": "user", "content": "Reply with exactly: kaggle works"}],
    max_tokens=20,
)
print(r.choices[0].message.content)

### Optional: public URL via Cloudflare (outbound-only, works on Kaggle)
Run in a terminal cell, then open the `*.trycloudflare.com` URL. **Set `LLAMA_API_KEY` in `.env` first** and restart the server, or anyone with the URL can use your GPU.
`cloudflared tunnel --url http://127.0.0.1:18123`

### Stop the server
`pkill -f llama-server`